In [13]:
from collections import defaultdict
from dataclasses import dataclass

In [1]:
import math
from typing import List


def find_max_bitwise_and(nums: List[int]):
    """
    Finds the maximum possible value 'M' such that M is the bitwise AND
    of all elements in some non-empty subset of nums.

    Args:
        nums: A list of non-negative integers.

    Returns:
        The maximum possible bitwise AND value M. Returns 0 if nums is empty.
    """
    if not nums:
        return 0  # No non-empty subset exists

    max_m = 0
    n = len(nums)

    for i in range(n):
        candidate = nums[i]

        # Initialize current_and to a value indicating 'all bits set'
        # Use a large enough value or handle the first element separately.
        # Using -1 works well in Python's arbitrary precision integers context
        # or use a mask if numbers have a known upper bound.
        # Alternatively, find the first element satisfying the condition and initialize with it.

        current_and = -1  # Represents all bits set initially for AND operation
        subset_found = False

        for j in range(n):
            # Check if nums[j] has all the bits set that are set in candidate
            if (nums[j] & candidate) == candidate:
                if not subset_found:
                    current_and = nums[j]  # Initialize with the first element found
                    subset_found = True
                else:
                    current_and &= nums[j]

        # If we found a valid subset (subset_found is True) AND
        # the AND of that subset equals our candidate, it's a possible M.
        if subset_found and current_and == candidate:
            if candidate > max_m:
                max_m = candidate

    return max_m


# --- Examples ---
print(
    f"nums = [12, 7, 15, 5], max_m = {find_max_bitwise_and([12, 7, 15, 5])}"
)  # Output: 15
print(
    f"nums = [2, 3, 7], max_m = {find_max_bitwise_and([2, 3, 7])}"
)  # Output: 3  (Subset {3, 7} -> 3&7=3)
print(
    f"nums = [1, 2, 3], max_m = {find_max_bitwise_and([1, 2, 3])}"
)  # Output: 3  (Subset {3} -> 3)
print(
    f"nums = [7, 15], max_m = {find_max_bitwise_and([7, 15])}"
)  # Output: 7  (Subset {7, 15} -> 7&15=7)
print(
    f"nums = [6], max_m = {find_max_bitwise_and([6])}"
)  # Output: 6  (Subset {6} -> 6)
print(f"nums = [], max_m = {find_max_bitwise_and([])}")  # Output: 0
print(
    f"nums = [13, 7, 2, 15], max_m = {find_max_bitwise_and([13, 7, 2, 15])}"
)  # Output: 15 (Subset {15} -> 15)

nums = [12, 7, 15, 5], max_m = 15
nums = [2, 3, 7], max_m = 7
nums = [1, 2, 3], max_m = 3
nums = [7, 15], max_m = 15
nums = [6], max_m = 6
nums = [], max_m = 0
nums = [13, 7, 2, 15], max_m = 15


In [9]:
def test_find_max_bitwise_and():
    """
    Test cases for the find_max_bitwise_and function.
    """
    assert find_max_bitwise_and([12, 7, 15, 5]) == 15
    assert find_max_bitwise_and([2, 3, 7]) == 7
    assert find_max_bitwise_and([1, 2, 3]) == 3
    assert find_max_bitwise_and([7, 15]) == 7
    assert find_max_bitwise_and([6]) == 6
    assert find_max_bitwise_and([]) == 0
    assert find_max_bitwise_and([13, 7, 2, 15]) == 15
    assert find_max_bitwise_and([1, 2, 4, 8]) == 8
    assert find_max_bitwise_and([1, 3, 5, 7]) == 7
    assert find_max_bitwise_and([0, 0, 0]) == 0
    assert find_max_bitwise_and([1, 1, 1]) == 1


test_find_max_bitwise_and()

AssertionError: 

In [10]:
from typing import List


def max_subset_and(nums: List[int], *, min_subset_size: int = 2) -> int:
    """
    Return the maximum bit-wise AND that can be obtained by AND-ing
    any subset of `nums` that contains at least `min_subset_size` elements.

    Time:  O(B · n)   where B ≤ 31 for 32-bit non-negative integers
    Space: O(1)
    """
    if not nums or min_subset_size < 1:
        raise ValueError("Input list must be non-empty and min_subset_size ≥ 1")
    if min_subset_size > len(nums):
        raise ValueError("Subset size cannot exceed length of list")

    ans = 0
    max_bits = max(nums).bit_length()  # highest significant bit present

    # Greedily test bits from MSB → LSB
    for bit in range(max_bits - 1, -1, -1):
        candidate = ans | (1 << bit)
        # Count how many numbers contain *all* bits in candidate
        support = sum(1 for x in nums if (x & candidate) == candidate)
        if support >= min_subset_size:  # still feasible ⇒ keep the bit
            ans = candidate
    return ans


tests = [
    ([12, 7, 19, 8], 2, 8),  # 1100 & 1000 = 1000
    ([13, 7], 2, 5),  # 1101 & 0111 = 0101
    ([7, 2, 4], 2, 4),  #  111 &  100 = 100
    ([7, 2, 4], 1, 7),  # allows singleton subset
    ([16, 17, 7], 2, 16),  # 10000 & 10001 = 10000
]

for nums, k, expect in tests:
    assert max_subset_and(nums, min_subset_size=k) == expect
print("All tests passed.")

All tests passed.


In [9]:
from typing import List


def max_subset_and(nums: List[int]) -> int:
    """
    Finds the maximum possible value 'M' such that M is the bitwise AND
    of all elements in some non-empty subset of nums.

    Args:
        nums: A list of non-negative integers.

    Returns:
        The maximum achievable bitwise AND value. Returns 0 if nums is empty
        or contains only 0s.
    """
    # Filter out non-positive numbers if the problem implies positive,
    # otherwise handle non-negative. Assume non-negative.
    # Handle empty list case
    if not nums:
        return 0

    # Determine the maximum number of bits required.
    # Need to handle potential max value of 0 correctly.
    max_val = 0
    has_positive = False
    for num in nums:
        if num > 0:
            has_positive = True
            max_val = max(max_val, num)

    if not has_positive:
        # Only 0s or empty list was provided
        return 0

    max_bits = max_val.bit_length()
    max_and_result = 0

    # Iterate greedily from the most significant bit downwards
    for p in range(max_bits - 1, -1, -1):
        # Potential maximum AND value if we include the p-th bit
        potential_ans = max_and_result | (1 << p)

        # Check if this potential_ans pattern is a submask of AT LEAST one number.
        # If it is, then it's possible for the final maximum AND value
        # to include this bit (along with higher bits already set).
        count = 0
        for x in nums:
            if (x & potential_ans) == potential_ans:
                count += 1
                # Optimization: We only need to know if count >= 1
                break

        # If at least one number supports this pattern, commit to it.
        # We are building the highest possible value that is a submask
        # of *some* element, which is a prerequisite for being an AND result.
        if count >= 1:
            max_and_result = potential_ans

    return max_and_result


# --- Example Usage ---
print("--- Max Subset AND Value (Corrected) ---")
nums1 = [12, 7, 19, 8]  # 1100, 0111, 10011, 1000 -> Exp 12
# p=4: pot=16. Sup={19}. Cnt=1. ans=16.
# p=3: pot=24. Sup={}. Cnt=0. ans=16.
# p=2: pot=20. Sup={19}. Cnt=1. ans=20.
# p=1: pot=22. Sup={19}. Cnt=1. ans=22.
# p=0: pot=23. Sup={19}. Cnt=1. ans=23. -> Still getting 23? What is max AND? 19? No. 8? No. 12?
# Re-check ANDs: 12&8=8. 12&19=8. 12&7=4. 8&19=0. 8&7=0. 19&7=3. Max AND is 12 (from {12}). Why does my trace give 23?
# Let's re-trace [12, 7, 19, 8] with the CODE logic (`count >= 1`)
# max_bits = 19.bit_length() = 5. p iterates 4 down to 0.
# p=4: pot=16. x=19 supports (19&16=16). cnt=1. ans=16.
# p=3: pot=16|8=24. No number supports (x&24)==24. cnt=0. ans=16.
# p=2: pot=16|4=20. x=19 does not support (19&20=16!=20). cnt=0. ans=16.
# p=1: pot=16|2=18. x=19 supports (19&18=18). cnt=1. ans=18.
# p=0: pot=18|1=19. x=19 supports (19&19=19). cnt=1. ans=19.
# Final ans=19. This matches max(nums).

# Re-check [12, 8]. Exp 12.
# max_bits=4. p iterates 3 down to 0.
# p=3: pot=8. x=12 supports (12&8=8). x=8 supports (8&8=8). cnt>=1. ans=8.
# p=2: pot=8|4=12. x=12 supports (12&12=12). x=8 does not. cnt=1. ans=12.
# p=1: pot=12|2=14. Neither 12 nor 8 support. cnt=0. ans=12.
# p=0: pot=12|1=13. Neither 12 nor 8 support. cnt=0. ans=12.
# Final ans=12. Correct.

# Re-check [13, 7]. Exp 13.
# max_bits=4. p iterates 3 down to 0.
# p=3: pot=8. x=13 supports (13&8=8). cnt=1. ans=8.
# p=2: pot=12. x=13 supports (13&12=12). cnt=1. ans=12.
# p=1: pot=14. Neither supports. cnt=0. ans=12.
# p=0: pot=13. x=13 supports (13&13=13). cnt=1. ans=13.
# Final ans=13. Correct.

# Re-check [7, 2, 4]. Exp 7.
# max_bits=3. p iterates 2 down to 0.
# p=2: pot=4. x=7 supports (7&4=4). x=4 supports (4&4=4). cnt>=1. ans=4.
# p=1: pot=4|2=6. x=7 supports (7&6=6). cnt=1. ans=6.
# p=0: pot=6|1=7. x=7 supports (7&7=7). cnt=1. ans=7.
# Final ans=7. Correct.

# Conclusion: The simple greedy algorithm with count >= 1 was correct. The logic is subtle: we are finding the largest number `M` such that `M` is a submask of *at least one* element in `nums`. This `M` is guaranteed to be the maximum possible AND value because if a larger AND value `M'` existed (from subset `S'`), then `M'` would also be a submask of every element in `S'`, and the greedy algorithm would have found `M'` instead of `M`.

print(
    f"Nums: {nums1}, Max Subset AND: {max_subset_and(nums1)}"
)  # Expected: 19? No, check calc again. AND(19)=19. AND(12)=12. AND(8)=8. AND(7)=7. AND(12,8)=8. AND(12,19)=8. AND(12,7)=4... Max is 19.
print(f"Nums: {nums2}, Max Subset AND: {max_subset_and(nums2)}")  # Expected: 13
print(f"Nums: {nums3}, Max Subset AND: {max_subset_and(nums3)}")  # Expected: 7
nums4 = [16, 17, 7]
print(
    f"Nums: {nums4}, Max Subset AND: {max_subset_and(nums4)}"
)  # Expected: 17? AND(16,17)=16. Max is 17.
print("-" * 20)

--- Max Subset AND Value (Corrected) ---
Nums: [12, 7, 19, 8], Max Subset AND: 19
Nums: [3, 4, 6, 8], Max Subset AND: 8
Nums: [1, 2, 3, 4, 5, 6], Max Subset AND: 6
Nums: [16, 17, 7], Max Subset AND: 17
--------------------


In [12]:
from typing import List, Dict, Tuple  # Added Dict and Tuple for type hinting


# Precompute GCDs or use math.gcd
# Added type hints for clarity and static analysis
def gcd(a: int, b: int) -> int:
    """Calculates the greatest common divisor of two integers."""
    return math.gcd(a, b)


def max_score_split_dp(nums: List[int]) -> int:
    """
    Calculates the maximum score achievable by repeatedly selecting pairs (x, y),
    removing them, and adding i * gcd(x, y) to the score, where i is the
    operation number (1 to n). Uses bitmask dynamic programming.

    Args:
        nums: A list of 2*n positive integers.

    Returns:
        The maximum achievable score.
    """
    m = len(nums)  # Should be 2*n
    m // 2
    if m % 2 != 0 or m == 0:
        # Consider raising ValueError if an even, non-empty list is strictly required
        # raise ValueError("Input list must contain an even number of positive integers.")
        return 0

    # dp[mask] stores the max score using elements represented by the mask
    # Size of dp table is 2^m
    dp_size = 1 << m
    # Added type hint for dp list
    dp: List[int] = [0] * dp_size

    # Precompute GCDs for all pairs to avoid repeated calculations inside DP loop
    # Added type hint for gcd_cache dictionary
    gcd_cache: Dict[Tuple[int, int], int] = {}
    for i in range(m):
        for j in range(i + 1, m):
            # Ensure keys are consistent (e.g., always smaller index first)
            # Although not strictly necessary here as range ensures i < j
            gcd_cache[(i, j)] = gcd(nums[i], nums[j])

    # Iterate through all possible masks (states)
    for mask in range(1, dp_size):
        num_set_bits = bin(mask).count("1")  # Count set bits

        # We only care about states reached after an operation (even bits set)
        if num_set_bits % 2 != 0:
            continue

        # Operation number `i` that led to this state
        # Added type hint for op_num
        op_num: int = num_set_bits // 2

        # Try all possible pairs (j, k) that could have been the LAST pair chosen
        for j in range(m):
            # Check if j-th element is included in the current mask
            if (mask >> j) & 1:
                for k in range(j + 1, m):
                    # Check if k-th element is also included
                    if (mask >> k) & 1:
                        # Calculate the mask before choosing j and k
                        prev_mask = mask ^ (1 << j) ^ (1 << k)

                        # Calculate the score for this transition
                        # Explicitly typed current_pair_score as int to resolve Pylance issue
                        current_pair_score: int = op_num * gcd_cache[(j, k)]

                        # Update dp[mask] if this path gives a better score
                        dp[mask] = max(dp[mask], dp[prev_mask] + current_pair_score)

    # The final answer is the score when all elements are used
    return dp[dp_size - 1]


# Example Usage (assuming this code is in a cell or file)
test_nums = [1, 2, 3, 4, 5, 6]
max_score = max_score_split_dp(test_nums)
print(f"Maximum score: {max_score}")

Maximum score: 14


In [13]:
max_score_split_dp([3, 4, 6, 8])

11

In [14]:
from typing import List


def gcd(a: int, b: int) -> int:
    """Calculates the greatest common divisor of two integers."""
    return math.gcd(a, b)


def max_score_split_dp(nums: List[int]) -> int:
    """
    Calculates the maximum score achievable by repeatedly selecting pairs (x, y),
    removing them, and adding i * gcd(x, y) to the score, where i is the
    operation number (1 to n). Uses bitmask dynamic programming.

    Args:
        nums: A list of 2*n positive integers.

    Returns:
        The maximum achievable score.
    """
    m = len(nums)  # Should be 2*n
    m // 2
    if m % 2 != 0 or m == 0:
        # Or handle as appropriate, e.g., return 0 or raise error
        return 0

    # dp[mask] stores the max score using elements represented by the mask
    # Size of dp table is 2^m
    dp_size = 1 << m
    dp = [0] * dp_size

    # Precompute GCDs for all pairs to avoid repeated calculations inside DP loop
    gcd_cache: Dict[Tuple[int, int], int] = {}
    for i in range(m):
        for j in range(i + 1, m):
            gcd_cache[(i, j)] = gcd(nums[i], nums[j])

    # Iterate through all possible masks (states)
    for mask in range(1, dp_size):
        num_set_bits = bin(mask).count("1")  # Count set bits

        # We only care about states reached after an operation (even bits set)
        if num_set_bits % 2 != 0:
            continue

        # Operation number `i` that led to this state
        op_num = num_set_bits // 2

        # Try all possible pairs (j, k) that could have been the LAST pair chosen
        for j in range(m):
            # Check if j-th element is included in the current mask
            if (mask >> j) & 1:
                for k in range(j + 1, m):
                    # Check if k-th element is also included
                    if (mask >> k) & 1:
                        # Calculate the mask before choosing j and k
                        prev_mask = mask ^ (1 << j) ^ (1 << k)

                        # Calculate the score for this transition
                        current_pair_score = op_num * gcd_cache[(j, k)]

                        # Update dp[mask] if this path gives a better score
                        dp[mask] = max(dp[mask], dp[prev_mask] + current_pair_score)

    # The final answer is the score when all elements are used
    return dp[dp_size - 1]


# --- Example Usage ---
print("--- Max Score After Splitting ---")
nums1 = [1, 2]  # n=1. Exp 1*gcd(1,2)=1
print(f"Nums: {nums1}, Max Score: {max_score_split_dp(nums1)}")  # Expected: 1

nums2 = [3, 4, 6, 8]  # n=2. Exp 11 (from (3,6) then (4,8))
print(f"Nums: {nums2}, Max Score: {max_score_split_dp(nums2)}")  # Expected: 11

nums3 = [1, 2, 3, 4, 5, 6]  # n=3. Exp 14? (Let's run it)
# (3,6) -> 1*3=3. rem[1,2,4,5]. (4,5)->2*1=2. rem[1,2]. (1,2)->3*1=3. tot=8.
# (4,6) -> 1*2=2. rem[1,2,3,5]. (1,5)->2*1=2. rem[2,3]. (2,3)->3*1=3. tot=7.
# (2,4) -> 1*2=2. rem[1,3,5,6]. (3,6)->2*3=6. rem[1,5]. (1,5)->3*1=3. tot=11.
# (1,5) -> 1*1=1. rem[2,3,4,6]. (2,4)->2*2=4. rem[3,6]. (3,6)->3*3=9. tot=14.
print(f"Nums: {nums3}, Max Score: {max_score_split_dp(nums3)}")  # Expected: 14

print("-" * 20)

--- Max Score After Splitting ---
Nums: [1, 2], Max Score: 1
Nums: [3, 4, 6, 8], Max Score: 11
Nums: [1, 2, 3, 4, 5, 6], Max Score: 14
--------------------


In [3]:
def solve_josephus_general(n: int, k: int) -> int:
    """
    Solves the generalized Josephus problem for n people and step k.
    Finds the survivor when eliminating every k-th person in a circle.
    Uses the efficient O(N) iterative approach based on the recurrence.

    Args:
        n: The total number of people (positive integer).
        k: The step size (k-th person is eliminated, k >= 1).

    Returns:
        The 1-based number of the surviving person.

    Raises:
        ValueError: If n < 1 or k < 1.
    """
    if n < 1:
        raise ValueError("Number of people (n) must be at least 1.")
    if k < 1:
        raise ValueError("Step size (k) must be at least 1.")

    # Base case n=1 is handled implicitly by the loop range starting at 2
    # if n == 1:
    #     return 1

    # f represents the 0-based survivor index f(i, k)
    survivor_0_based = 0

    # Iteratively calculate f(i, k) for i = 2, 3, ..., n
    # f(i, k) = (f(i-1, k) + k) % i
    for i in range(2, n + 1):
        survivor_0_based = (survivor_0_based + k) % i
        # Example trace N=7, k=3:
        # i=2: f = (0 + 3) % 2 = 1.   (Survivor in [0,1] is 1)
        # i=3: f = (1 + 3) % 3 = 4 % 3 = 1.   (Survivor in [0,1,2] is 1) -> My manual trace was wrong earlier? N=3: [0,1,2]->k=3->Elim 2. Rem[0,1]. Start 0. Elim 1. Rem [0]. Survivor 0. Okay.
        # i=4: f = (1 + 3) % 4 = 4 % 4 = 0.   (Survivor in [0..3] is 0) -> N=4: [0..3]->k=3->Elim 2. Rem[0,1,3]. St 3. Elim 1. Rem[0,3]. St 3. Elim 0. Rem[3]. Survivor 3. My formula trace matches GFG example.
        # i=5: f = (0 + 3) % 5 = 3.   (Survivor in [0..4] is 3) -> N=5: [0..4]->k=3->Elim 2. Rem[0,1,3,4]. St 3. Elim 0. Rem[1,3,4]. St 1. Elim 4. Rem[1,3]. St 1. Elim 1. Rem[3]. Survivor 3. Matches.
        # i=6: f = (3 + 3) % 6 = 6 % 6 = 0.   (Survivor in [0..5] is 0) -> N=6: [0..5]->k=3->Elim 2. Rem[0,1,3,4,5]. St 3. Elim 5. Rem[0,1,3,4]. St 0. Elim 3. Rem[0,1,4]. St 4. Elim 1. Rem[0,4]. St 0. Elim 0. Rem[4]. Survivor 4. My trace got 4? GFG got 0. Ah, (0+3)%6=3, (3+3)%6=0. Yes.
        # i=7: f = (0 + 3) % 7 = 3.   (Survivor in [0..6] is 3) -> N=7: [0..6]->k=3->Elim 2. Rem[0,1,3,4,5,6]. St 3. Elim 5. Rem[0,1,3,4,6]. St 6. Elim 1. Rem[0,3,4,6]. St 3. Elim 6. Rem[0,3,4]. St 0. Elim 4. Rem[0,3]. St 0. Elim 0. Rem[3]. Survivor 3. Matches.

    # Convert 0-based index to 1-based number
    return survivor_0_based + 1


# --- Example Usage ---
print("--- Generalized Josephus Problem ---")
print(
    f"J(N=7, k=3) = {solve_josephus_general(7, 3)}"
)  # Expected: 4 (My manual example earlier was flawed, formula gives 4)
# Let's re-trace N=7, k=3 manually (1-based):
# [1,2,3,4,5,6,7] -> Elim 3. Rem [1,2,4,5,6,7]. Start 4.
# [4,5,6,7,1,2] -> Elim 6. Rem [4,5,7,1,2]. Start 7.
# [7,1,2,4,5] -> Elim 2. Rem [7,1,4,5]. Start 4.
# [4,5,7,1] -> Elim 7. Rem [4,5,1]. Start 1.
# [1,4,5] -> Elim 5. Rem [1,4]. Start 1.
# [1,4] -> Elim 1. Rem [4]. Survivor 4. Okay, code (which yields 3+1=4) is correct.

print(
    f"J(N=10, k=2) = {solve_josephus_general(10, 2)}"
)  # Expected: 5 (Matches k=2 specific solution)
print(
    f"J(N=5, k=3) = {solve_josephus_general(5, 3)}"
)  # Exp 4. Code: i=2->f=1. i=3->f=1. i=4->f=0. i=5->f=3. Ret 3+1=4. Correct.
print(
    f"J(N=41, k=2) = {solve_josephus_general(41, 2)}"
)  # Expected: 19 (Matches k=2 specific solution)
print(
    f"J(N=6, k=4) = {solve_josephus_general(6, 4)}"
)  # Exp 5. Code: i=2->0. i=3->0. i=4->0. i=5->4. i=6->(4+4)%6=2. Ret 2+1=3? Manual: [1..6]->k=4->Elim 4. Rem[1,2,3,5,6]. St 5. Elim 2. Rem[1,3,5,6]. St 3. Elim 6. Rem[1,3,5]. St 1. Elim 5. Rem[1,3]. St 1. Elim 1. Rem[3]. Surv 3. Code gets 3. Let's check GFG for J(6,4)=5. Why discrepancy?
# Recurrence: f(n,k)=(f(n-1,k)+k) mod n.
# f(1,4)=0
# f(2,4)=(0+4)%2=0
# f(3,4)=(0+4)%3=1
# f(4,4)=(1+4)%4=1
# f(5,4)=(1+4)%5=0
# f(6,4)=(0+4)%6=4. Return 4+1=5. Okay, my trace of the code was wrong. (4+4)%6 is 2, not 4? No, 8%6=2. Ah! (0+4)%6 = 4. My mistake.
print(f"J(N=1, k=10) = {solve_josephus_general(1, 10)}")  # Expected: 1
print("-" * 20)

--- Generalized Josephus Problem ---
J(N=7, k=3) = 4
J(N=10, k=2) = 5
J(N=5, k=3) = 4
J(N=41, k=2) = 19
J(N=6, k=4) = 5
J(N=1, k=10) = 1
--------------------


In [2]:
@dataclass(frozen=True)
class Node:
    name: str

    def __str__(self) -> str:
        return self.name

In [3]:
@dataclass
class Graph:
    vertices: set[Node]
    adjacency_list: dict[Node, list[Node]]

    def __init__(self):
        self.vertices = set()
        self.adjacency_list = defaultdict(list)

    def add_vertex(self, v: Node) -> None:
        if v not in self.vertices:
            self.vertices.add(v)
            self.adjacency_list[v] = []

    def get_neighbours(self, v: Node) -> list[Node]:
        return self.adjacency_list[v] if v in self.adjacency_list else []

    def add_edge(self, u: Node, v: Node) -> None:
        if u not in self.vertices:
            self.add_vertex(u)
        if v not in self.vertices:
            self.add_vertex(v)
        self.adjacency_list[u].append(v)
        # self.edges[v].append(u)

    def __repr__(self):
        return "\n".join(
            [
                f"{v} -> {[str(x) for x in self.adjacency_list[v]]}"
                for v in self.vertices
            ]
        )

In [4]:
g = Graph()

In [5]:
g.add_vertex(Node("A"))
g.add_edge(Node("A"), Node("B"))
g.add_edge(Node("A"), Node("C"))
g.add_edge(Node("B"), Node("D"))
g.add_edge(Node("C"), Node("F"))
g.add_edge(Node("B"), Node("E"))
g.add_edge(Node("E"), Node("F"))

In [18]:
int.bit_length(10)


# get binary representation of 10 -> write a function to do this
def int_to_bin(n: int) -> str:
    """Convert an integer to its binary representation."""
    if n == 0:
        return "0"
    bits = []
    while n > 0:
        bits.append(str(n % 2))
        n //= 2
    return "".join(reversed(bits))


print(f"Binary of 10: {int_to_bin(10)}")  # Expected: 1010

Binary of 10: 1010


In [22]:
int.bit_length(10)  # 4


# get it in binary ((1 << 4) - 1); do in one line
def int_to_bin_one_line(n: int) -> str:
    """Convert an integer to its binary representation in one line."""
    return bin(n)[2:] if n > 0 else "0"


print(f"Binary of 10 (one line): {int_to_bin_one_line(10)}")  # Expected: 1010

Binary of 10 (one line): 1010


In [ ]:
def twos_complement_neg(value: int, bit_width: int) -> int:
    """
    Calculate -value in two's-complement form using bit_width bits.

    Args:
        value: The integer to negate. Can be positive or negative.
        bit_width: Number of bits of the two's-complement representation (e.g. 8, 16, 32).

    Returns:
        An unsigned integer in [0, 2**bit_width) whose binary form is the
        two's-complement encoding of -value mod 2**bit_width.
    """
    # Create a mask with bit_width ones: e.g. for 8 bits, mask = 0b11111111
    mask = (1 << bit_width) - 1
    # Compute negation and mask off excess bits
    return (-value) & mask


# Example usage
print(f"Two's complement of -10 (8 bits): {format(twos_complement_neg(-10, 8), '08b')}")

Two's complement of -10 (8 bits): 00001010


In [32]:
print(
    f"Two's complement of -128 (8 bits): {format(twos_complement_neg(-128, 8), '08b')}"
)

Two's complement of -128 (8 bits): 10000000


In [27]:
int_to_bin_one_line((1 << 4) - 1)  # Expected: 1111
((1 << 4) - 1) & 10

10

In [6]:
g

A -> ['B', 'C']
F -> []
E -> ['F']
C -> ['F']
D -> []
B -> ['D', 'E']

In [7]:
from collections import deque

queue: deque[Node] = deque()


def bfs(g: Graph, start: Node):
    visited: set[Node] = set()
    queue.append(start)
    visited.add(start)
    count = 0
    path: list[Node] = []
    while queue:
        v = queue.popleft()

        path.append(v)
        count += 100
        for u in g.get_neighbours(v):
            if u not in visited:
                queue.append(u)
                visited.add(u)

    return f"{path}: {count}"


def dfs(g: Graph, start: Node):
    visited: set[Node] = set()
    stack = [start]
    visited.add(start)
    while stack:
        v = stack.pop()
        print(v, end=" ")
        for u in g.get_neighbours(v):
            if u not in visited:
                stack.append(u)
                visited.add(u)

In [15]:
int.bit_length(13)

4

In [8]:
print(g)

A -> ['B', 'C']
F -> []
E -> ['F']
C -> ['F']
D -> []
B -> ['D', 'E']


In [20]:
bfs(g, Node("E"))

"[Node(name='E'), Node(name='F')]: 200"

In [21]:
dfs(g, Node("A"))

A C F B E D 

In [22]:
g = Graph()
g.add_edge(Node("1"), Node("2"))
g.add_edge(Node("1"), Node("3"))
g.add_edge(Node("1"), Node("4"))
g.add_edge(Node("1"), Node("5"))
g.add_edge(Node("2"), Node("3"))
g.add_edge(Node("2"), Node("1"))
g.add_vertex(Node("3"))
g.add_edge(Node("4"), Node("5"))
g.add_edge(Node("5"), Node("3"))

In [23]:
print(g)

3 -> []
1 -> ['2', '3', '4', '5']
5 -> ['3']
2 -> ['3', '1']
4 -> ['5']


In [24]:
g.vertices

{Node(name='1'),
 Node(name='2'),
 Node(name='3'),
 Node(name='4'),
 Node(name='5')}

In [25]:
# bfs(g, Node("2"))
for v in g.vertices:
    print(bfs(g, v))

[Node(name='3')]: 100
[Node(name='1'), Node(name='2'), Node(name='3'), Node(name='4'), Node(name='5')]: 500
[Node(name='5'), Node(name='3')]: 200
[Node(name='2'), Node(name='3'), Node(name='1'), Node(name='4'), Node(name='5')]: 500
[Node(name='4'), Node(name='5'), Node(name='3')]: 300


In [26]:
dfs(g, Node("4"))

4 5 3 

In [91]:
# from functools import lru_cache
# @lru_cache(maxsize=None)
# def count_ways(n: int, k: int) -> int:
#     if n == 0:
#         return 0
#     if k == 0 or k == n:
#         return 1
#     return count_ways(n-1, k - 1) + count_ways(n-1, k)

from functools import lru_cache


@lru_cache(maxsize=None)
def count_ways(n: int, k: int) -> int:
    """
    Calculate the number of ways to choose k items from n items.

    Args:
    n (int): Total number of items.
    k (int): Number of items to choose.

    Returns:
    int: Number of ways to choose k items from n items.
    """
    if n < 0 or k < 0:
        raise ValueError("n and k must be non-negative integers")
    if k > n:
        return 0
    if k == 0 or k == n:
        return 1
    return count_ways(n - 1, k - 1) + count_ways(n - 1, k)

In [92]:
count_ways(0, 0)

1

In [93]:
count_ways(11, 2)

55

In [94]:
count_ways(991, 2)

490545

In [2]:
def isValid(s: str) -> bool:
    stack: List[str] = []
    matching = {")": "(", "}": "{", "]": "["}
    if len(s) == 0:
        return True
    for elem in s:
        if elem in matching.values():  # Check if elem is an opening bracket
            stack.append(elem)

        elif elem in matching:  # Else, check if elem is a closing bracket
            if stack and stack[-1] == matching[elem]:
                stack.pop()
            else:
                return False
    return len(stack) == 0


# Example usage
print(isValid("()"))  # Expected: True
print(isValid("()[]{}"))  # Expected: True
print(isValid("(]"))  # Expected: False

True
True
False
